In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from haversine import haversine_vector, Unit
import itertools
import requests, zipfile, io

**city-city // postal code-postal code distance**

In [30]:
# data from BokanyiE
irsz_data = pd.read_csv("../data/iranyitoszamok_BE.csv")
irsz_data["irsz"] = irsz_data["irsz"].astype(int)
irsz_data = irsz_data[["irsz", "lon", "lat"]].drop_duplicates(subset="irsz")

In [34]:
# dataframe w/ full combinations
pcodes = irsz_data["irsz"].unique()
full_comb = pd.MultiIndex.from_product(
    [pcodes, pcodes], names=["pcode1", "pcode2"]
)
full_comb = pd.DataFrame(index=full_comb).reset_index().sort_values(by=["pcode1", "pcode2"])
print(full_comb.shape)

(9278116, 2)


In [35]:
# add coords
full_comb = pd.merge(
    full_comb,
    irsz_data[["irsz", "lon", "lat"]].drop_duplicates(),
    left_on="pcode1",
    right_on="irsz",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    irsz_data[["irsz", "lon", "lat"]].drop_duplicates(),
    left_on="pcode2",
    right_on="irsz",
    how="left",
    suffixes=["1", "2"]
)
full_comb.drop(columns=["pcode1", "pcode2"], inplace=True)
print(full_comb.shape)

(9278116, 6)


In [37]:
# distance calculation
full_comb["coords1"] = list(
    zip(full_comb["lat1"], full_comb["lon1"])
)
full_comb["coords2"] = list(
    zip(full_comb["lat2"], full_comb["lon2"])
)
full_comb["distance"] = haversine_vector(
    full_comb["coords1"].tolist(), full_comb["coords2"].tolist()
)

In [49]:
# export
full_comb = full_comb[["irsz1", "irsz2", "distance"]]
full_comb["distance"] = full_comb["distance"].astype(int)
export_full_comb = full_comb[full_comb["irsz1"] < full_comb["irsz2"]]
export_full_comb.to_csv("../outputs/pcode_pcode_distance.csv", index=False, sep=";")

In [25]:
# alternative -- settlements table from BokanyiE
set_df = pd.read_csv("../data/settlements.csv")

In [26]:
set_df

,NAME,postal_code,eov_x,eov_y,POPULATION
0,Tótszerdahely,8864,476836.854045,119430.851538,1045.0
1,Molnári,8863,479972.644358,117850.809377,705.0
2,Semjénháza,8862,481013.043220,119883.048881,579.0
3,Felsőszölnök,9985,430353.770993,174339.396387,579.0
4,Lendvadedes,8978,453328.134780,145006.205485,28.0
...,...,...,...,...,...
3313,Tiszabecs,4951,930844.729899,312930.743417,1412.0
3314,Garbolc,4976,935254.907542,296326.693832,139.0
3315,Magosliget,4953,934186.695439,307845.056084,346.0
3316,Uszka,4952,933286.338570,310014.784856,486.0


**NUTS4-NUTS4 distance**

In [39]:
# read geography
rs_shape = gpd.read_file("../data/shape_files/nuts4_shape.geojson")


In [40]:
# drop duplicates -- Encsi jaras -- choose the larger area 
rs_shape["area"] = rs_shape.area
idx = rs_shape.groupby(["region_name"])["area"].transform(max) == rs_shape["area"]
rs_shape = rs_shape[idx]

In [41]:
# centroid coords
rs_shape = rs_shape.to_crs("epsg:4326")
rs_shape["lon"] = rs_shape["geometry"].centroid.x
rs_shape["lat"] = rs_shape["geometry"].centroid.y

/var/folders/9d/8j37_fks51x11mk0_zwqsd940000gn/T/ipykernel_12738/1759052024.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  rs_shape["lon"] = rs_shape["geometry"].centroid.x
/var/folders/9d/8j37_fks51x11mk0_zwqsd940000gn/T/ipykernel_12738/1759052024.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  rs_shape["lat"] = rs_shape["geometry"].centroid.y


In [42]:
# create full region-region combination
nuts4_name = rs_shape["region_name"].unique()
full_comb = pd.MultiIndex.from_product(
    [nuts4_name, nuts4_name], names=["region1", "region2"]
)
full_comb = pd.DataFrame(index=full_comb).reset_index().sort_values(by=["region1", "region2"])

In [43]:
# add coords
full_comb = pd.merge(
    full_comb,
    rs_shape[["region_name", "lon", "lat"]].drop_duplicates(),
    left_on="region1",
    right_on="region_name",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    rs_shape[["region_name", "lon", "lat"]].drop_duplicates(),
    left_on="region2",
    right_on="region_name",
    how="left",
    suffixes=["1", "2"]
)
full_comb.drop(columns=["region_name1", "region_name2"], inplace=True)

In [44]:
# distance calculation
full_comb["coords1"] = list(
    zip(full_comb["lat1"], full_comb["lon1"])
)
full_comb["coords2"] = list(
    zip(full_comb["lat2"], full_comb["lon2"])
)
full_comb["distance"] = haversine_vector(
    full_comb["coords1"].tolist(), full_comb["coords2"].tolist()
)

In [45]:
# ksh region codes
region_codes = pd.read_csv("../outputs/region_names_codes.csv", sep=";")

In [46]:
# for KSH incheck
full_comb = pd.merge(
    full_comb[["region1", "region2", "distance"]],
    region_codes[["nuts4_code", "nuts4_name"]].drop_duplicates(),
    left_on="region1",
    right_on="nuts4_name",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    region_codes[["nuts4_code", "nuts4_name"]].drop_duplicates(),
    left_on="region2",
    right_on="nuts4_name",
    how="left",
    suffixes=["1", "2"]
)
full_comb.rename(columns={"nuts4_code1":"region_code1", "nuts4_code2":"region_code2"}, inplace=True)
full_comb.drop(columns=["nuts4_name1", "nuts4_name2"], inplace=True)

In [47]:
# export
full_comb.to_csv("../outputs/nuts4_nuts4_distance.csv", index=False, sep=";")